# Conformational control of HOMO–VBM alignment in carbazole 2D perovskites

This notebook merges SATool structural descriptors with VASP-derived HOMO, VBM and CBM energies for `cz-c_3`, `cz-c_4`, and `cz-c_5`. It reconstructs Ward clusters from ammonium anchoring descriptors, tests structural–electronic correlations, fits interpretable and nonlinear models, and performs a pooled analysis.

**Scientific caution:** compare eigenvalues across structures only after using a common energy reference (e.g. planar-averaged electrostatic potential, core-level reference, or vacuum reference). A small HOMO–VBM offset identifies an energetic resonance condition; demonstrate actual hybridization using PDOS, band projections, and band-decomposed charge densities.

In [ ]:
from __future__ import annotations
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.stats import spearmanr, linregress, bootstrap
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.decomposition import PCA
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('results')
OUTPUT_DIR = Path('analysis_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_WARD_CLUSTERS = 5
RANDOM_STATE = 42

# Adjust only these paths if your files use another naming convention.
COMPOUND_PATHS = {
    'cz-c_3': {'satool': DATA_DIR / 'satool_props_c3.csv', 'dft': DATA_DIR / 'c_3-dft-eigenvalues.dat'},
    'cz-c_4': {'satool': DATA_DIR / 'satool_props_c4.csv', 'dft': DATA_DIR / 'c_4-dft-eigenvalues.dat'},
    'cz-c_5': {'satool': DATA_DIR / 'satool_props_c5.csv', 'dft': DATA_DIR / 'c_5-dft-eigenvalues.dat'},
}

# If the DFT file uses nonstandard headers, add the exact names here.
DFT_ALIASES = {
    'structure_id': ['structure_id', 'structure', 'id', 'config_id', 'configuration_id', 'filename', 'file', 'name', 'index'],
    'vbm': ['vbm', 'e_vbm', 'vbm_ev', 'vbm_energy', 'valence_band_maximum'],
    'homo': ['homo', 'e_homo', 'homo_ev', 'homo_energy', 'organic_homo'],
    'cbm': ['cbm', 'e_cbm', 'cbm_ev', 'cbm_energy', 'conduction_band_minimum'],
}

SELECTION_RULES = {
    'trans_angle': {'keywords': ['trans', 'angle'], 'prefer': ['mean'], 'avoid': ['std', 'min', 'max', 'n']},
    'shared_mxm_angle': {'keywords': ['shared', 'halide', 'angle'], 'prefer': ['mean', 'in_plane'], 'avoid': ['std', 'min', 'max', 'n']},
    'chain_tail_dist': {'keywords': ['chain', 'tail'], 'prefer': ['mean', 'inorganic', 'plane', 'dist'], 'avoid': ['std', 'min', 'max', 'n']},
    'tilt_fragment': {'keywords': ['whole_fragment', 'inorganic_layer_plane'], 'prefer': ['mean'], 'avoid': ['std', 'min', 'max']},
    'tilt_core': {'keywords': ['ring_core', 'inorganic_layer_plane'], 'prefer': ['mean'], 'avoid': ['std', 'min', 'max']},
    'tilt_chain': {'keywords': ['chain_principal', 'inorganic_layer_plane'], 'prefer': ['mean'], 'avoid': ['std', 'min', 'max']},
    'hook_elevation': {'keywords': ['hook', 'elevation'], 'prefer': ['abs_mean', 'inorganic', 'plane'], 'avoid': ['std', 'min', 'max', 'n']},
}


## Analysis functions
All reusable functions are defined here. The following cells apply the same workflow first to `cz-c_3`, then `cz-c_4`, `cz-c_5`, and finally to the combined dataset.

In [ ]:
def normalise_name(value):
    return re.sub(r'[^a-z0-9]+', '_', str(value).lower()).strip('_')

def read_table_auto(path):
    if not Path(path).exists():
        raise FileNotFoundError(f'File not found: {Path(path).resolve()}')
    try:
        df = pd.read_csv(path, sep=None, engine='python', comment='#')
    except Exception:
        df = pd.read_csv(path, sep=r'\s+', engine='python', comment='#')
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=r'\s+', engine='python', comment='#')
    return df

def canonicalize_id(values):
    text = values.astype(str).str.strip()
    number = text.str.extract(r'(\d+)(?!.*\d)')[0]
    return np.where(number.notna(), number.fillna('').str.lstrip('0').replace('', '0'), text)

def find_column(columns, aliases, required=True):
    normalized = {normalise_name(c): c for c in columns}
    for alias in aliases:
        if normalise_name(alias) in normalized:
            return normalized[normalise_name(alias)]
    matches = []
    for alias in aliases:
        matches.extend([original for norm, original in normalized.items() if normalise_name(alias) in norm])
    matches = list(dict.fromkeys(matches))
    if len(matches) == 1:
        return matches[0]
    if required:
        raise KeyError(f'Could not identify a required column. Tried {aliases}. Available: {list(columns)}')
    return None

def score_satool_column(column, rule):
    name = normalise_name(column)
    if any(word not in name for word in rule['keywords']):
        return -np.inf
    return (10 * len(rule['keywords'])
            + 3 * sum(word in name for word in rule.get('prefer', []))
            - 10 * sum(word in name for word in rule.get('avoid', [])))

def infer_satool_columns(df):
    mapping = {}
    for canonical, rule in SELECTION_RULES.items():
        candidates = {c: score_satool_column(c, rule) for c in df.columns}
        candidates = {c: s for c, s in candidates.items() if np.isfinite(s)}
        mapping[canonical] = max(candidates, key=candidates.get) if candidates else None
        print(f'{canonical:20s} <- {mapping[canonical]}')
    return mapping

def load_satool(path):
    raw = read_table_auto(path)
    raw.columns = [str(c).strip() for c in raw.columns]
    id_col = find_column(raw.columns, DFT_ALIASES['structure_id'], required=False)
    raw['structure_id'] = canonicalize_id(raw[id_col]) if id_col else np.arange(len(raw)).astype(str)
    if id_col is None:
        print('WARNING: SATool structure identifier not found; using row index.')
    mapping = infer_satool_columns(raw)
    valid = {key: value for key, value in mapping.items() if value is not None}
    out = raw[['structure_id', *valid.values()]].rename(columns={v: k for k, v in valid.items()}).copy()
    for col in valid:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    return out

def load_dft(path):
    raw = read_table_auto(path)
    raw.columns = [str(c).strip() for c in raw.columns]
    selected = {name: find_column(raw.columns, aliases) for name, aliases in DFT_ALIASES.items()}
    out = raw[list(selected.values())].rename(columns={v: k for k, v in selected.items()}).copy()
    out['structure_id'] = canonicalize_id(out['structure_id'])
    for col in ['vbm', 'homo', 'cbm']:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    return out

def add_electronic_metrics(df):
    out = df.copy()
    # Positive delta_homo_vbm means HOMO is above VBM with the conventional energy axis.
    out['delta_homo_vbm'] = out['homo'] - out['vbm']
    out['abs_delta_homo_vbm'] = out['delta_homo_vbm'].abs()
    out['inorganic_gap'] = out['cbm'] - out['vbm']
    out['effective_gap'] = out['cbm'] - np.maximum(out['homo'], out['vbm'])
    out['alignment_regime'] = pd.cut(out['delta_homo_vbm'], [-np.inf, -0.05, 0.15, np.inf],
                                     labels=['VBM_above_HOMO', 'near_resonance', 'type_II_separated'])
    return out

def assign_ward_clusters(df, n_clusters=N_WARD_CLUSTERS):
    out = df.copy()
    features = ['chain_tail_dist', 'hook_elevation']
    valid = out.dropna(subset=features).copy()
    if len(valid) < n_clusters:
        raise ValueError('Not enough complete structures to form the requested Ward clusters.')
    scaled = StandardScaler().fit_transform(valid[features])
    labels = fcluster(linkage(scaled, method='ward'), t=n_clusters, criterion='maxclust')
    out['ward_cluster'] = np.nan
    out.loc[valid.index, 'ward_cluster'] = labels.astype(int)
    return out

def correlation_table(df, features, targets):
    rows = []
    for target in targets:
        for feature in features:
            subset = df[[feature, target]].dropna()
            rho, p = spearmanr(subset[feature], subset[target]) if len(subset) >= 5 else (np.nan, np.nan)
            rows.append({'feature': feature, 'target': target, 'n': len(subset), 'spearman_rho': rho, 'pvalue': p})
    out = pd.DataFrame(rows)
    out['pvalue_fdr_bh'] = np.nan
    valid = out['pvalue'].notna()
    out.loc[valid, 'pvalue_fdr_bh'] = multipletests(out.loc[valid, 'pvalue'], method='fdr_bh')[1]
    return out.sort_values(['target', 'pvalue_fdr_bh'])

def bootstrap_slope(x, y, n_resamples=3000):
    data = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(data) < 6:
        return {'n': len(data), 'slope': np.nan, 'low': np.nan, 'high': np.nan}
    stat = lambda xx, yy, axis=-1: linregress(xx, yy).slope
    ci = bootstrap((data.x.to_numpy(), data.y.to_numpy()), stat, paired=True, n_resamples=n_resamples,
                   random_state=RANDOM_STATE, method='percentile').confidence_interval
    return {'n': len(data), 'slope': linregress(data.x, data.y).slope, 'low': ci.low, 'high': ci.high}

def fit_ols(df, target, features):
    usable = [f for f in features if f in df.columns]
    data = df[[target, *usable]].dropna()
    if len(data) < len(usable) + 10:
        return None
    terms = ' + '.join(usable)
    if 'chain_tail_dist' in usable and 'hook_elevation' in usable:
        terms += ' + chain_tail_dist:hook_elevation'
    return smf.ols(f'{target} ~ {terms}', data=data).fit(cov_type='HC3')

def fit_nonlinear_model(df, target, features):
    data = df[[target, *features]].dropna()
    if len(data) < 20:
        return None, None, None
    model = ExtraTreesRegressor(n_estimators=1000, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(data[features], data[target])
    perm = permutation_importance(model, data[features], data[target], n_repeats=100,
                                  random_state=RANDOM_STATE, scoring='neg_mean_absolute_error', n_jobs=-1)
    importance = pd.DataFrame({'feature': features, 'importance_mae': perm.importances_mean,
                               'importance_std': perm.importances_std}).sort_values('importance_mae', ascending=False)
    cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
    mae = -cross_val_score(model, data[features], data[target], scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
    return model, importance, mae

def plot_compound(df, compound, corr, importance=None):
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    sns.scatterplot(data=df, x='chain_tail_dist', y='hook_elevation', hue='ward_cluster', palette='tab10', s=85, ax=axes[0])
    axes[0].set_title(f'{compound}: Ward sampling space')
    sc = axes[1].scatter(df['chain_tail_dist'], df['vbm'], c=df['hook_elevation'], cmap='viridis', s=85, edgecolor='k', linewidth=.25)
    sns.regplot(data=df, x='chain_tail_dist', y='vbm', scatter=False, color='crimson', ax=axes[1])
    axes[1].set_title(f'{compound}: VBM vs ammonium distance')
    fig.colorbar(sc, ax=axes[1], label='Hook elevation')
    sc2 = axes[2].scatter(df['chain_tail_dist'], df['hook_elevation'], c=df['delta_homo_vbm'], cmap='coolwarm', s=85, edgecolor='k', linewidth=.25)
    axes[2].set_title(f'{compound}: HOMO–VBM alignment map')
    fig.colorbar(sc2, ax=axes[2], label='HOMO - VBM (eV)')
    fig.tight_layout()
    plt.show()
    heat = corr.pivot(index='feature', columns='target', values='spearman_rho')
    plt.figure(figsize=(10, 6))
    sns.heatmap(heat, cmap='vlag', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f')
    plt.title(f'{compound}: Spearman structural–electronic correlations')
    plt.tight_layout(); plt.show()
    if importance is not None:
        plt.figure(figsize=(8, 5))
        sns.barplot(data=importance, x='importance_mae', y='feature', color='steelblue')
        plt.title(f'{compound}: nonlinear importance for VBM')
        plt.tight_layout(); plt.show()

def representative_structures(df, n_per_regime=5):
    selected = []
    for regime, group in df.groupby('alignment_regime', observed=False):
        group = group.dropna(subset=['abs_delta_homo_vbm'])
        if not group.empty:
            out = group.nsmallest(n_per_regime, 'abs_delta_homo_vbm').copy()
            out['selection_reason'] = f'lowest_offset_{regime}'
            selected.append(out)
    columns = ['compound', 'structure_id', 'ward_cluster', 'chain_tail_dist', 'hook_elevation',
               'vbm', 'homo', 'cbm', 'delta_homo_vbm', 'alignment_regime', 'selection_reason']
    return pd.concat(selected, ignore_index=True)[columns] if selected else pd.DataFrame(columns=columns)

def run_compound(compound, paths):
    satool = load_satool(paths['satool'])
    dft = load_dft(paths['dft'])
    df = satool.merge(dft, on='structure_id', how='inner', validate='one_to_one')
    df['compound'] = compound
    df = add_electronic_metrics(df)
    df = assign_ward_clusters(df)
    print(f'\n{compound}: SATool={len(satool)}, DFT={len(dft)}, merged={len(df)}')
    if len(df) < .8 * len(dft):
        print('WARNING: fewer than 80% of DFT rows matched; inspect structure IDs.')
    features = [f for f in SELECTION_RULES if f in df.columns]
    targets = ['vbm', 'homo', 'cbm', 'delta_homo_vbm', 'abs_delta_homo_vbm', 'inorganic_gap', 'effective_gap']
    corr = correlation_table(df, features, targets)
    display(df[targets].describe().T)
    display(corr.sort_values('pvalue_fdr_bh').head(20))
    slope = bootstrap_slope(df['chain_tail_dist'], df['vbm'])
    print(f"VBM vs chain-tail distance slope: {slope['slope']:.4f} eV/unit; 95% CI [{slope['low']:.4f}, {slope['high']:.4f}], n={slope['n']}")
    ols_vbm = fit_ols(df, 'vbm', features)
    ols_offset = fit_ols(df, 'delta_homo_vbm', features)
    if ols_vbm is not None:
        print('\nOLS: VBM'); print(ols_vbm.summary())
    if ols_offset is not None:
        print('\nOLS: HOMO–VBM offset'); print(ols_offset.summary())
    _, importance, cv_mae = fit_nonlinear_model(df, 'vbm', features)
    if cv_mae is not None:
        print(f'ExtraTrees VBM repeated CV MAE: {cv_mae.mean():.3f} ± {cv_mae.std():.3f} eV')
        display(importance)
    plot_compound(df, compound, corr, importance)
    reps = representative_structures(df)
    display(reps)
    safe = compound.replace('-', '_')
    df.to_csv(OUTPUT_DIR / f'{safe}_merged_structural_dft.csv', index=False)
    corr.to_csv(OUTPUT_DIR / f'{safe}_correlations.csv', index=False)
    reps.to_csv(OUTPUT_DIR / f'{safe}_representative_structures.csv', index=False)
    if importance is not None:
        importance.to_csv(OUTPUT_DIR / f'{safe}_vbm_feature_importance.csv', index=False)
    return {'data': df, 'correlations': corr, 'ols_vbm': ols_vbm, 'ols_offset': ols_offset, 'importance': importance, 'representatives': reps}


## Optional input check
Run this cell first if automatic column detection fails. Confirm that each DFT file has one row per structure and recognizable identifier, VBM, HOMO, and CBM headers.

In [ ]:
for compound, paths in COMPOUND_PATHS.items():
    print(f'\n--- {compound} ---')
    for kind, path in paths.items():
        try:
            preview = read_table_auto(path)
            print(f'{kind}: {path} | columns = {list(preview.columns)}')
            display(preview.head(3))
        except FileNotFoundError as exc:
            print(exc)


## `cz-c_3` analysis
This is the central system for testing whether conformations with larger ammonium-plane distances bring the inorganic VBM closer to the organic HOMO.

In [ ]:
results_c3 = run_compound('cz-c_3', COMPOUND_PATHS['cz-c_3'])
df_c3 = results_c3['data']


## `cz-c_4` analysis

In [ ]:
results_c4 = run_compound('cz-c_4', COMPOUND_PATHS['cz-c_4'])
df_c4 = results_c4['data']


## `cz-c_5` analysis

In [ ]:
results_c5 = run_compound('cz-c_5', COMPOUND_PATHS['cz-c_5'])
df_c5 = results_c5['data']


## Pooled analysis: `cz-c_3`, `cz-c_4`, and `cz-c_5`
The interaction `chain_tail_dist × compound` tests whether the anchoring–VBM relation is universal or chain-length dependent.

In [ ]:
all_compounds = pd.concat([df_c3, df_c4, df_c5], ignore_index=True)
pooled_features = ['chain_tail_dist', 'hook_elevation', 'shared_mxm_angle', 'trans_angle', 'tilt_core', 'tilt_chain']
required = ['vbm', 'delta_homo_vbm', 'compound', *pooled_features]
pooled = all_compounds.dropna(subset=required).copy()

vbm_model = smf.ols(
    'vbm ~ chain_tail_dist * C(compound) + hook_elevation + shared_mxm_angle + trans_angle + tilt_core + tilt_chain',
    data=pooled
).fit(cov_type='HC3')
offset_model = smf.ols(
    'delta_homo_vbm ~ chain_tail_dist * C(compound) + hook_elevation + shared_mxm_angle + trans_angle + tilt_core + tilt_chain',
    data=pooled
).fit(cov_type='HC3')

print('Pooled VBM model'); print(vbm_model.summary())
print('\nPooled HOMO–VBM offset model'); print(offset_model.summary())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=all_compounds, x='chain_tail_dist', y='vbm', hue='compound', style='compound', s=90, ax=axes[0])
for compound, group in all_compounds.groupby('compound'):
    sns.regplot(data=group, x='chain_tail_dist', y='vbm', scatter=False, ax=axes[0])
axes[0].set_title('VBM versus ammonium anchoring')
sns.scatterplot(data=all_compounds, x='chain_tail_dist', y='delta_homo_vbm', hue='compound', style='alignment_regime', s=90, ax=axes[1])
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].axhspan(-0.05, 0.15, color='gold', alpha=.18)
axes[1].set_title('HOMO–VBM offset across all compounds')
fig.tight_layout(); plt.show()

slopes = pd.DataFrame([{'compound': compound, **bootstrap_slope(group['chain_tail_dist'], group['vbm'])}
                       for compound, group in all_compounds.groupby('compound')])
display(slopes)
all_compounds.to_csv(OUTPUT_DIR / 'all_compounds_merged_structural_dft.csv', index=False)
slopes.to_csv(OUTPUT_DIR / 'all_compounds_vbm_distance_slopes.csv', index=False)
(OUTPUT_DIR / 'pooled_vbm_model.txt').write_text(vbm_model.summary().as_text())
(OUTPUT_DIR / 'pooled_homo_vbm_offset_model.txt').write_text(offset_model.summary().as_text())


## Select structures for higher-level electronic calculations
The exported table contains low-offset structures from each alignment regime. Prioritize thermodynamically accessible structures (include MACE relative energies if available) and calculate HSE06+SOC, PDOS, orbital projections, and band-decomposed charge densities.

In [ ]:
all_representatives = pd.concat([results_c3['representatives'], results_c4['representatives'], results_c5['representatives']], ignore_index=True)
display(all_representatives)
all_representatives.to_csv(OUTPUT_DIR / 'representative_structures_for_HSE_SOC_PDOS.csv', index=False)
print(f'Analysis tables were written to: {OUTPUT_DIR.resolve()}')
